# Fine Print — heavy baselines on Kaggle (v0.3)

Runs the jobs the Mac can't handle, **on both free T4 GPUs at the same time**:

| GPU | Jobs (in order) |
|---|---|
| 0 | B1 zero-shot Qwen2.5-3B on `test-500`, then on `lexglue` |
| 1 | B3 DeBERTa-v3-small (train + all eval sets), then B2 3-shot Qwen on `test-500`, then on `lexglue` |

**Setup (once):**
1. On kaggle.com: *Datasets → New dataset* → upload `dist/kaggle/fine-print-bundle.zip` → keep it **Private**.
2. *Code → New notebook → File → Import notebook* → this file.
3. Right panel: *Add input* → your dataset · *Accelerator* → **GPU T4 x2** · *Internet* → **on**.
4. *Save Version → Save & Run All (Commit)*. It runs in the background (≈2–3 h), so you can close the tab.
5. When it's done: open the version → *Output* → download `fine-print-results.zip` and put it in the repo root.
   Then on the Mac: `uv run python -m fineprint.kaggle_results`.


In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import time

# Find the uploaded bundle (Kaggle unpacks zips) and copy it somewhere writable.
src = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/fine-print/configs', recursive=True))
REPO = '/kaggle/working/fine-print'
if not os.path.exists(REPO):
    shutil.copytree(src, REPO)
os.chdir(REPO)
ENV = {**os.environ, 'PYTHONPATH': f'{REPO}/src', 'TOKENIZERS_PARALLELISM': 'false'}
print(REPO, os.listdir(REPO))
subprocess.run(['nvidia-smi', '-L'])

In [ ]:
# Quick check that the code and data load before the long runs start.
subprocess.run([sys.executable, '-c',
    'from fineprint.eval.sets import load_eval_set as l; print({s: len(l(s)) for s in ("test-500", "lexglue")})'],
    env=ENV, check=True)

In [ ]:
JOBS = {
    0: ['python -m fineprint.eval.run_hf --system b1_qwen3b_zeroshot --set test-500',
        'python -m fineprint.eval.run_hf --system b1_qwen3b_zeroshot --set lexglue'],
    1: ['python -m fineprint.baselines.deberta --epochs 2',
        'python -m fineprint.eval.run_hf --system b2_qwen3b_fewshot --set test-500 --shots 3',
        'python -m fineprint.eval.run_hf --system b2_qwen3b_fewshot --set lexglue --shots 3'],
}
procs = {}
for gpu, cmds in JOBS.items():
    log = open(f'/kaggle/working/gpu{gpu}.log', 'w')
    # one shell per GPU: its jobs run one after another; a failed job doesn't stop the next
    procs[gpu] = subprocess.Popen(['bash', '-c', '; '.join(cmds)], stdout=log, stderr=subprocess.STDOUT,
                                  env={**ENV, 'CUDA_VISIBLE_DEVICES': str(gpu)})
print('started', {g: p.pid for g, p in procs.items()})

In [ ]:
# Wait for both GPUs, printing progress every 2 minutes.
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    for gpu in procs:
        lines = open(f'/kaggle/working/gpu{gpu}.log').read().strip().splitlines()
        print(f'gpu{gpu}:', lines[-1][:150] if lines else '(starting)')
print({g: p.returncode for g, p in procs.items()})

In [ ]:
# Package what the Mac needs: scores, predictions and logs (no model weights).
import zipfile

with zipfile.ZipFile('/kaggle/working/fine-print-results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['results/eval/*.json', 'data/predictions/*/*.jsonl']:
        for f in glob.glob(f'{REPO}/{pattern}'):
            z.write(f, os.path.relpath(f, REPO))
    for f in glob.glob('/kaggle/working/gpu*.log'):
        z.write(f, 'logs/kaggle_' + os.path.basename(f))
print(zipfile.ZipFile('/kaggle/working/fine-print-results.zip').namelist())